# APGI Quick-Start

This notebook walks through the core APGI equations, runs a short trial sequence,
and visualises the global integration signal $S_t$ alongside the adaptive threshold $\theta_t$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import apgi
from apgi.integration import APGICoreIntegration

print(f"apgi version: {apgi.__version__}")

## 1 — Core equations

The three fundamental quantities are:

| Symbol | Formula | Meaning |
|--------|---------|----------|
| $\Pi^i_{\text{eff}}$ | $\Pi^i \cdot e^{-C / \kappa}$ | Metabolically gated inhibitory precision |
| $S_t$ | $\Pi^e \|z^e\| + \Pi^i_{\text{eff}} \|z^i\|$ | Global integration signal |
| $\theta_t$ | $\alpha C + \beta V$ | Adaptive ignition threshold |

In [ ]:
pi_i_eff = apgi.compute_pi_i_eff(pi_i=1.0, C_metabolic=50.0, kappa=100.0)
print(f"Pi_i_eff at C=50, kappa=100: {pi_i_eff:.4f}  (expect ~{np.exp(-0.5):.4f})")

S_t = apgi.compute_S_t(pi_e=1.2, z_e=0.8, pi_i_eff=pi_i_eff, z_i=0.5)
theta_t = apgi.compute_theta_t(C_metabolic=1.0, V_information=0.5, alpha=0.3, beta=0.7)
print(
    f"S_t = {S_t:.4f}   theta_t = {theta_t:.4f}   ignition = {apgi.ignition_criterion(S_t, theta_t)}"
)

## 2 — Running a session with `APGICoreIntegration`

`APGICoreIntegration` manages the rolling $\theta_t$ update and accumulates trial records.

In [ ]:
rng = np.random.default_rng(0)
n = 200

pi_e = rng.uniform(0.8, 1.5, n)
z_e = rng.uniform(0.2, 1.0, n)
pi_i = rng.uniform(0.5, 1.5, n)
z_i = rng.uniform(0.1, 0.8, n)
C_metabolic = rng.uniform(0.5, 2.0, n)
V_information = rng.uniform(0.1, 1.0, n)

integ = APGICoreIntegration(alpha=0.3, beta=0.7, kappa=100.0, gamma=0.9)
records = integ.run_sequence(pi_e, z_e, pi_i, z_i, C_metabolic, V_information)

print(f"Ignition rate: {integ.ignition_rate():.3f}")
print(f"Mean S_t:      {integ.mean_S_t():.3f}")
print(f"Mean theta_t:  {integ.mean_theta():.3f}")

## 3 — Visualising ignition dynamics

In [ ]:
S_series = np.array([r.S_t for r in records])
theta_series = np.array([r.theta_t for r in records])
fired = np.array([r.ignition for r in records])
t = np.arange(n)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, S_series, lw=1.5, color="#2166ac", label=r"$S_t$")
ax.plot(t, theta_series, lw=1.5, color="#d6604d", ls="--", label=r"$\theta_t$")
ax.vlines(t[fired], 0, S_series.max(), colors="#f4a582", alpha=0.4, lw=0.8)
ax.set_xlabel("Trial")
ax.set_ylabel("Signal")
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 4 — Signal normalisation across sessions

`APGINormalizer` fits a z-score or min-max scaler on one session and applies it to another.

In [ ]:
from apgi.normalizer import APGINormalizer

norm = APGINormalizer(method="zscore")
S_normed = norm.fit_transform(S_series)
print(f"After normalisation:  mean={S_normed.mean():.2e}  std={S_normed.std():.4f}")